# Count All SI Sorted Neurons

In [6]:
from analyses.spike_count import filter_good_neurons
import os
import pandas as pd

In [7]:
folder = "/home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache"
total_neurons = 0
for file in os.listdir(folder):

    if not file.endswith(".pkl"):
        continue

    path = os.path.join(folder, file)

    df = pd.read_pickle(path)
    num_neuron = df['NeuronID'].nunique()
    total_neurons += num_neuron

print(total_neurons)

697


# Filter good neurons

In [10]:
from analyses.spike_count import filter_good_neurons
import os
import pandas as pd

input_folder = "/home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache"
output_folder = "/home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):

    if not file.endswith(".pkl"):
        continue

    path = os.path.join(input_folder, file)

    df = pd.read_pickle(path)

    # Per-condition coverage instead of a total-trial floor: require every one of
    # the 37 photos to have >= 6 reps (all 37 present). This targets the balanced
    # coverage that correlation-distance RDM/RSA and state-space trajectories need
    # directly, rather than a session total that can hide a starved photo.
    # min_trial_count is left off (=1); the per-condition rule is the active gate.
    filtered_list = filter_good_neurons(df,
                                        min_total_spikes=300,
                                        min_firing_rate_hz=1,
                                        min_trial_count=1,
                                        n_time_blocks=4,
                                        min_active_blocks= 3,
                                        max_isi_violation_rate=0.02,
                                        refractory_ms=2.0,
                                        min_reps_per_condition=6,
                                        n_conditions=37
                                        )

    if len(filtered_list) == 0:
        print(f"Skipping {file} (no neurons passed filter)")
        continue
    filtered_df = df[df["NeuronID"].isin(filtered_list)]

    save_path = os.path.join(output_folder, file)
    filtered_df.to_pickle(save_path)
    # print(filtered_df)
    print(f"Saved filtered file: {save_path}")

Skipping 2023-09-29_round_4.pkl (no neurons passed filter)
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-12-07_round_3.pkl
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-09-29_round_3.pkl
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-10-27_round_4.pkl
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-12-18_round_3.pkl
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-12-11_round_2.pkl
Skipping 2023-10-05_round_1.pkl (no neurons passed filter)
Skipping 2023-10-31_round_3.pkl (no neurons passed filter)
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-11-22_round_3.pkl
Saved filtered file: /home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered/2023-12-18_ro

# Count the total number of neurons that passed the filtering

In [11]:
# Count the total number of

folder = "/home/connorlab/Documents/GitHub/Julie/Cortana/sorted_spike_cache_filtered"

total_neurons = 0
total_ER_neurons = 0
total_AMG_neurons = 0
total_Unknown_neurons = 0
for file in os.listdir(folder):
    if not file.endswith(".pkl"):
        continue

    path = os.path.join(folder, file)

    df = pd.read_pickle(path)

    n_neurons = df["NeuronID"].nunique()
    ER_neurons = df.loc[df["NeuronID"].str.contains("ER"), "NeuronID"].nunique()
    AMG_neurons = df.loc[df["NeuronID"].str.contains("AMG"), "NeuronID"].nunique()
    Unknown_neurons = df.loc[df["NeuronID"].str.contains("Unknown"), "NeuronID"].nunique()

    total_ER_neurons += ER_neurons
    total_AMG_neurons += AMG_neurons
    total_neurons += n_neurons
    total_Unknown_neurons += Unknown_neurons
    # print(f"{file}: {n_neurons} neurons")

print("Total neurons across all files:", total_neurons)
print("Total ER neurons across all files:", total_ER_neurons)
print("Total AMG neurons across all files:", total_AMG_neurons)
print("Total Unknown_neurons across all files:", total_Unknown_neurons)

Total neurons across all files: 320
Total ER neurons across all files: 166
Total AMG neurons across all files: 119
Total Unknown_neurons across all files: 35
